## 1. Imports and configuration

In [ ]:
# If needed, uncomment in Google Colab/Jupyter:
#%pip install -q opencv-python seaborn scikit-learn tqdm plotly

import os
import random
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from PIL import Image
from tqdm.auto import tqdm

RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["image.interpolation"] = "nearest"

print("OpenCV:", cv2.__version__)
print("NumPy:", np.__version__)
print("Pandas:", pd.__version__)

In [ ]:
# drive.mount('/content/drive')  # from google.colab import drive
# DATASET_DIR = Path('/content/drive/MyDrive/CIFAKE')
DATASET_DIR = Path(r"D:\JKU\2026,27 Winter\AI & VIs\CIFAKE")

TRAIN_DIR = DATASET_DIR / "train"
TEST_DIR = DATASET_DIR / "test"  # inspected only for counts, not feature exploration

CLASS_NAMES = ["REAL", "FAKE"]
VALID_EXTENSIONS = {".png", ".jpg", ".jpeg", ".bmp", ".webp"}

if not TRAIN_DIR.exists():
    raise FileNotFoundError(
        f"Training directory not found: {TRAIN_DIR}\n"
        "Edit DATASET_DIR so it points to the CIFAKE folder."
    )

print("Training directory:", TRAIN_DIR.resolve())
print("Test directory exists:", TEST_DIR.exists())

## 3. Index files and validate the folder structure

In [ ]:
def list_class_images(split_dir, class_names=CLASS_NAMES):
    records = []
    for label in class_names:
        class_dir = split_dir / label
        if not class_dir.is_dir():
            print(f"WARNING: missing class folder: {class_dir}")
            continue
        for path in sorted(class_dir.rglob("*")):
            if path.is_file() and path.suffix.lower() in VALID_EXTENSIONS:
                records.append({"label": label, "image_path": str(path)})
    return pd.DataFrame(records)

train_df = list_class_images(TRAIN_DIR)
if train_df.empty:
    raise RuntimeError("No images found. Check DATASET_DIR and class folder names.")

print("Training images:", len(train_df))
display(train_df.groupby("label").size().rename("image_count").to_frame())
print("\nFile extensions:")
display(train_df.image_path.map(lambda p: Path(p).suffix.lower()).value_counts().to_frame("count"))

## 4. Class distribution

In [ ]:
class_counts = train_df["label"].value_counts().reindex(CLASS_NAMES, fill_value=0)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
sns.barplot(x=class_counts.index, y=class_counts.values, ax=axes[0], hue=class_counts.index, legend=False)
axes[0].set(title="Training images by class", xlabel="Class", ylabel="Number of images")
for i, value in enumerate(class_counts.values):
    axes[0].text(i, value, f"{value:,}", ha="center", va="bottom")

axes[1].pie(class_counts.values, labels=class_counts.index, autopct="%1.1f%%", startangle=90)
axes[1].set_title("Class proportions")
plt.tight_layout()
plt.show()

print("Class balance ratio (minority / majority):",
      round(class_counts.min() / class_counts.max(), 4) if class_counts.max() else "N/A")

## 5. Display a reproducible sample grid

In [ ]:
def show_sample_grid(dataframe, n_per_class=5, seed=RANDOM_SEED):
    rng = np.random.default_rng(seed)
    fig, axes = plt.subplots(len(CLASS_NAMES), n_per_class,
                             figsize=(2.5*n_per_class, 5.5), squeeze=False)
    for row, label in enumerate(CLASS_NAMES):
        subset = dataframe[dataframe.label == label]
        n = min(n_per_class, len(subset))
        if n == 0:
            continue
        chosen = rng.choice(subset.index.to_numpy(), size=n, replace=False)
        for col in range(n_per_class):
            ax = axes[row, col]
            ax.axis("off")
            if col < n:
                path = subset.loc[chosen[col], "image_path"]
                with Image.open(path) as im:
                    ax.imshow(im.convert("RGB"))
                ax.set_title(label if col == 0 else "")
    fig.suptitle("Example CIFAKE training images", y=1.02)
    plt.tight_layout()
    plt.show()

show_sample_grid(train_df, n_per_class=6)

In [ ]:
MAX_IMAGES_PER_CLASS = 10000

def extract_features(dataframe, max_per_class=MAX_IMAGES_PER_CLASS, seed=RANDOM_SEED):
    sampled = (
        dataframe.groupby("label", group_keys=False)
        .sample(n=min(max_per_class, dataframe.groupby("label").size().min()),
                random_state=seed)
        .sort_values(["label", "image_path"])
    )
    rows = []
    for row in tqdm(sampled.itertuples(index=False), total=len(sampled), desc="Extracting features"):
        bgr = cv2.imread(row.image_path, cv2.IMREAD_COLOR)
        if bgr is None:
            continue
        rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
        rgb_f = rgb.astype(np.float32) / 255.0
        gray = cv2.cvtColor(rgb, cv2.COLOR_RGB2GRAY).astype(np.float32) / 255.0
        hsv = cv2.cvtColor(rgb, cv2.COLOR_RGB2HSV).astype(np.float32)

        gx = cv2.Sobel(gray, cv2.CV_32F, 1, 0, ksize=3)
        gy = cv2.Sobel(gray, cv2.CV_32F, 0, 1, ksize=3)
        edge_strength = np.mean(np.sqrt(gx**2 + gy**2))

        rows.append({
            "label": row.label,
            "image_path": row.image_path,
            "height": rgb.shape[0],
            "width": rgb.shape[1],
            "brightness": float(gray.mean()),
            "contrast": float(gray.std()),
            "red_mean": float(rgb_f[:, :, 0].mean()),
            "green_mean": float(rgb_f[:, :, 1].mean()),
            "blue_mean": float(rgb_f[:, :, 2].mean()),
            "saturation": float((hsv[:, :, 1] / 255.0).mean()),
            "edge_density": float(edge_strength),
        })
    return pd.DataFrame(rows)

features_df = extract_features(train_df)
print("Feature rows:", len(features_df))
display(features_df.head())
display(features_df.groupby("label").size().rename("sampled_images").to_frame())

## 7. Brightness and contrast distributions

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, feature, title, xlabel in [
    (axes[0], "brightness", "Brightness distribution", "Mean grayscale intensity (0–1)"),
    (axes[1], "contrast", "Contrast distribution", "Grayscale standard deviation (0–1)")
]:
    sns.histplot(data=features_df, x=feature, hue="label", bins=40,
                 stat="density", common_norm=False, element="step",
                 kde=False, ax=ax)
    ax.set_title(title)
    ax.set_xlabel(xlabel)
    ax.set_ylabel("Density")
plt.tight_layout()
plt.show()

## 8. RGB channel and saturation distributions

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 9))
for ax, feature, title in [
    (axes[0, 0], "red_mean", "Mean red intensity"),
    (axes[0, 1], "green_mean", "Mean green intensity"),
    (axes[1, 0], "blue_mean", "Mean blue intensity"),
    (axes[1, 1], "saturation", "Mean HSV saturation"),
]:
    sns.histplot(data=features_df, x=feature, hue="label", bins=35,
                 stat="density", common_norm=False, element="step", ax=ax)
    ax.set_title(title)
    ax.set_xlabel("Normalized value (0–1)")
    ax.set_ylabel("Density")
plt.tight_layout()
plt.show()

## 9. Brightness versus contrast

In [ ]:
plot_df = features_df.sample(n=min(5000, len(features_df)), random_state=RANDOM_SEED)
plt.figure(figsize=(9, 6))
sns.scatterplot(data=plot_df, x="brightness", y="contrast",
                hue="label", alpha=0.45, s=22)
plt.title("Brightness versus contrast")
plt.xlabel("Mean grayscale intensity")
plt.ylabel("Grayscale standard deviation")
plt.tight_layout()
plt.show()

## 10. Visualize extreme examples

In [ ]:
def show_extremes(dataframe, feature, n=3):
    fig, axes = plt.subplots(2, n, figsize=(3*n, 6), squeeze=False)
    for col, label in enumerate(CLASS_NAMES):
        subset = dataframe[dataframe.label == label]
        if subset.empty:
            continue
        # Top row: lowest values; bottom row: highest values
        selected = [subset.nsmallest(1, feature).iloc[0],
                    subset.nlargest(1, feature).iloc[0]]
        for row_idx, record in enumerate(selected):
            ax = axes[row_idx, col] if n == 1 else axes[row_idx, col]
            ax.imshow(Image.open(record.image_path).convert("RGB"))
            ax.set_title(f"{label} | {feature}={record[feature]:.3f}")
            ax.axis("off")
    plt.tight_layout()
    plt.show()

# One darkest and one brightest example per class
show_extremes(features_df, "brightness")
# One lowest-contrast and one highest-contrast example per class
show_extremes(features_df, "contrast")

## 11. Average images and absolute difference heatmap


In [ ]:
def average_class_image(dataframe, label, max_images=5000, seed=RANDOM_SEED):
    subset = dataframe[dataframe.label == label]
    subset = subset.sample(n=min(max_images, len(subset)), random_state=seed)
    accumulator = None
    count = 0
    for path in tqdm(subset.image_path, desc=f"Averaging {label}"):
        image = cv2.imread(path)
        if image is None:
            continue
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB).astype(np.float32) / 255.0
        if accumulator is None:
            accumulator = np.zeros_like(image, dtype=np.float64)
        if image.shape != accumulator.shape:
            continue
        accumulator += image
        count += 1
    if count == 0:
        raise ValueError(f"No readable, consistently sized images for {label}")
    return (accumulator / count).astype(np.float32)

avg_real = average_class_image(train_df, "REAL")
avg_fake = average_class_image(train_df, "FAKE")
abs_diff = np.abs(avg_real - avg_fake)

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
axes[0].imshow(avg_real); axes[0].set_title("Mean REAL image")
axes[1].imshow(avg_fake); axes[1].set_title("Mean FAKE image")
im = axes[2].imshow(abs_diff.mean(axis=2), cmap="inferno")
axes[2].set_title("Mean absolute RGB difference")
fig.colorbar(im, ax=axes[2], fraction=0.046, label="Absolute difference")
for ax in axes: ax.axis("off")
plt.tight_layout()
plt.show()

## 12. Fourier frequency analysis


In [ ]:
def average_fourier_spectrum(dataframe, label, sample_size=500, seed=RANDOM_SEED):
    subset = dataframe[dataframe.label == label]
    subset = subset.sample(n=min(sample_size, len(subset)), random_state=seed)
    spectra = []
    for path in tqdm(subset.image_path, desc=f"FFT {label}"):
        gray = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
        if gray is None:
            continue
        spectrum = np.fft.fftshift(np.fft.fft2(gray.astype(np.float32)))
        spectra.append(20 * np.log(np.abs(spectrum) + 1e-8))
    if not spectra:
        raise ValueError(f"No readable images for {label}")
    return np.mean(spectra, axis=0)

fft_real = average_fourier_spectrum(train_df, "REAL")
fft_fake = average_fourier_spectrum(train_df, "FAKE")

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
vmin = min(fft_real.min(), fft_fake.min())
vmax = max(fft_real.max(), fft_fake.max())
axes[0].imshow(fft_real, cmap="magma", vmin=vmin, vmax=vmax); axes[0].set_title("REAL mean log spectrum")
axes[1].imshow(fft_fake, cmap="magma", vmin=vmin, vmax=vmax); axes[1].set_title("FAKE mean log spectrum")
diff = np.abs(fft_real - fft_fake)
im = axes[2].imshow(diff, cmap="inferno"); axes[2].set_title("Absolute spectrum difference")
fig.colorbar(im, ax=axes[2], fraction=0.046)
for ax in axes: ax.axis("off")
plt.tight_layout()
plt.show()

## 13. Edge-strength distribution

In [ ]:
plt.figure(figsize=(9, 5))
sns.violinplot(data=features_df, x="label", y="edge_density", hue="label",
               palette="muted", inner="quartile", legend=False)
plt.title("Sobel edge strength by class")
plt.xlabel("Class")
plt.ylabel("Mean gradient magnitude")
plt.tight_layout()
plt.show()

## 14. Optional: PCA and t-SNE on flattened pixels


In [ ]:
# Optional dependency:
%pip install -q scikit-learn

from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.preprocessing import StandardScaler

PCA_TSNE_PER_CLASS = 500

balanced = (
    train_df.groupby("label", group_keys=False)
    .sample(n=min(PCA_TSNE_PER_CLASS, train_df.groupby("label").size().min()),
            random_state=RANDOM_SEED)
)

pixel_vectors = []
pixel_labels = []
for row in tqdm(balanced.itertuples(index=False), total=len(balanced), desc="Loading pixels"):
    image = cv2.imread(row.image_path)
    if image is None:
        continue
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    if image.shape[:2] != (32, 32):
        image = cv2.resize(image, (32, 32), interpolation=cv2.INTER_AREA)
    pixel_vectors.append(image.astype(np.float32).reshape(-1) / 255.0)
    pixel_labels.append(row.label)

X = np.asarray(pixel_vectors)
y = np.asarray(pixel_labels)

# PCA to 50 dimensions (or fewer if the sample is small)
n_components = min(50, X.shape[0] - 1, X.shape[1])
X_pca = PCA(n_components=n_components, random_state=RANDOM_SEED).fit_transform(X)

# t-SNE perplexity must be smaller than the number of samples
perplexity = min(30, max(5, (len(X_pca) - 1) // 3))
X_tsne = TSNE(n_components=2, perplexity=perplexity,
              init="pca", learning_rate="auto",
              random_state=RANDOM_SEED).fit_transform(X_pca)

tsne_df = pd.DataFrame({"tSNE-1": X_tsne[:, 0],
                        "tSNE-2": X_tsne[:, 1],
                        "label": y})
plt.figure(figsize=(9, 6))
sns.scatterplot(data=tsne_df, x="tSNE-1", y="tSNE-2",
                hue="label", alpha=0.65, s=25)
plt.title("t-SNE of CIFAKE image pixels")
plt.tight_layout()
plt.show()

## 15. Numerical summary and export


In [ ]:
feature_columns = [
    "brightness", "contrast", "red_mean", "green_mean",
    "blue_mean", "saturation", "edge_density"
]
summary = features_df.groupby("label")[feature_columns].agg(["count", "mean", "std", "median"])
display(summary)

features_df.to_csv("cifake_eda_features.csv", index=False)
summary.to_csv("cifake_eda_summary.csv")
print("Saved cifake_eda_features.csv and cifake_eda_summary.csv")